# Explicabilidad para atribución de autoría con un transformer entrenado usando Captum

Este notebook está preparado para ser ejecutado en Google Colab puesto que se necesita GPU para la inferencia.

Hace lo ssguiente:

- Carga un modelo `AutoModelForSequenceClassification` ya entrenado.
- Carga su tokenizer y su `config` con `id2label` / `label2id`.
- Predice el autor para una obra.
- Calcula explicabilidad con *Integrated Gradients* usando **Captum**.
- Visualiza la importancia de los tokens para la clase predicha o una clase objetivo.
- Agrega predicciones por chunks para documentos largos.

## Captum

Captum es una librería *open source* de explicabilidad e interpretabilidad para modelos desarrollados en PyTorch. Su objetivo es ayudar a entender por qué un modelo produce una determinada predicción, identificando qué variables de entrada, neuronas o capas internas han contribuido más al resultado. Para ello, ofrece una interfaz unificada para aplicar métodos de atribución como *Integrated Gradients*, *DeepLift*, *GradientSHAP*, *Feature Ablation* o *Guided Grad-CAM*, entre otros. Además, está diseñada para trabajar con distintos tipos de datos, como imágenes, texto o datos tabulares, y puede integrarse con modelos PyTorch existentes con pocas modificaciones.

En este notebook se utiliza Captum para realizar un ejercicio de explicabilidad sobre el modelo de clasificación de textos de tipo *transformer* que previamente hemos entrenado en el notebook `10_train_authorship_transformer_colab.ipynb`. En concreto, el objetivo es analizar por qué el modelo predice que una determinada obra o fragmento de texto pertenece a un cieto autor.

Para ello se emplea el método **Layer Integrated Gradients**, una variante de *Integrated Gradients* que permite calcular atribuciones no solo respecto a la entrada original, sino también respecto a una capa interna del modelo, como puede ser la capa de *embeddings* en un modelo de lenguaje. Esto resulta especialmente útil en tareas de procesamiento de lenguaje natural, ya que permite estimar qué tokens, palabras o fragmentos del texto han contribuido más a la predicción de autor realizada por el modelo.

De esta forma, Captum no se utiliza únicamente para obtener una explicación global del funcionamiento del clasificador, sino para interpretar predicciones individuales. Las atribuciones generadas por **LayerIntegratedGradients** permiten identificar qué partes del texto han tenido mayor influencia positiva o negativa en la asignación de una obra a un determinado autor, aportando una visión más transparente del comportamiento del modelo y facilitando el análisis de posibles patrones estilísticos aprendidos durante el entrenamiento.

**Documentación oficial de Captum:** <https://captum.ai/docs/introduction>

In [1]:
!pip -q install -U "numpy>=2,<2.1" "transformers" "captum" "scikit-learn" "pandas==2.2.2"

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import json
from pathlib import Path
from typing import Optional, Any, Sequence

import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from captum.attr import LayerIntegratedGradients

print("CUDA disponible:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

CUDA disponible: True
Dispositivo: cuda


## Configuración

In [4]:
MODEL_DIR = "/content/drive/MyDrive/outputs/authorship_transformer"
TEXT_PATH = Path("/content/drive/MyDrive/corpus/arthur_conan_doyle/acd-the_hound_of_the_baskervilles.txt")
# TEXT_PATH = Path("/content/drive/MyDrive/corpus/arthur_conan_doyle/acd-the_lost_world.txt")
PASTED_TEXT = None  # Podemos pegar aquí un fragmento texto en lugar de usar una obra cargada

MAX_LENGTH = 256
STRIDE = 64
TOP_CHUNKS_TO_EXPLAIN = 3

# Con TARGET_LABEL = None explica la clase predicha
TARGET_LABEL = None
ALTERNATIVE_LABEL = "anna_katharine_green"

## Carga del modelo y del tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(DEVICE)
model.eval()

id2label = model.config.id2label
if isinstance(next(iter(id2label.keys())), str):
    id2label = {int(k): v for k, v in id2label.items()}
label2id = model.config.label2id

print("Modelo cargado desde:", MODEL_DIR)
print("Etiquetas:", id2label)
print("Modelo en:", next(model.parameters()).device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo cargado desde: /content/drive/MyDrive/outputs/authorship_transformer
Etiquetas: {0: 'anna_katharine_green', 1: 'arthur_conan_doyle', 2: 'arthur_morrison', 3: 'gilbert_keith_chesterton', 4: 'richard_austin_freeman', 5: 'wilkie_collins'}
Modelo en: cuda:0


## Funciones auxiliares

Dado que este *notebook* se ejecutará en Colab, es necesario definir algunas funciones auxiliares para cargar los datos y preparar el dataset para el entrenamiento. Estas funciones se han adaptado para funcionar con la estructura de archivos en Colab.

A diferencia de los *notebooks* anteriores, donde las utilidades estaban organizadas en módulos separados, aquí se incluyen directamente en el *notebook* para facilitar su ejecución sin necesidad de importar desde otros archivos.

In [6]:
# @title Funciones auxiliares
def get_text(text_path: Optional[Path] = None, pasted_text: Optional[object] = None) -> Optional[str]:
    """Return stripped text from pasted input or a text file.

    Pasted text takes precedence over file input. If `pasted_text` is not `None`
    and contains non-whitespace characters after conversion to `str`, the
    stripped pasted text is returned. Otherwise, if `text_path` is provided,
    the function reads text from that path and returns it stripped.

    Args:
        text_path: Path to a text file to read when usable pasted text is not
            provided.
        pasted_text: Optional pasted content. It is converted to `str` before
            whitespace trimming.

    Returns:
        The stripped text from `pasted_text` or `text_path`, or `None` if neither
        source provides text.
    """
    if pasted_text is not None and str(pasted_text).strip():
        return str(pasted_text).strip()
    if text_path is not None:
        return text_path.read_text().strip()
    return None


def chunk_document(
    text: str,
    tokenizer: Any,
    max_length: int = 256,
    stride: int = 64,
) -> list[dict[str, list[int] | list[tuple[int, int]]]]:
    """Split text into overlapping tokenized chunks.

    Tokenizes `text` using the provided tokenizer and returns one dictionary per
    produced chunk. Each chunk contains token IDs, an attention mask, and offset
    mappings. Overflowing tokens are returned, so long inputs may produce
    multiple chunks with overlap controlled by `stride`.

    Args:
        text: Text to tokenize and split into chunks.
        tokenizer: Tokenizer callable that supports Hugging Face-style tokenizer
            arguments and returns mappings for `input_ids`, `attention_mask`,
            and `offset_mapping`.
        max_length: Maximum number of tokens per chunk, including special
            tokens.
        stride: Number of overlapping tokens to keep between consecutive chunks
            when overflowing tokens are returned.

    Returns:
        A list of chunk dictionaries. Each dictionary contains:
            - `input_ids`: Token IDs for the chunk.
            - `attention_mask`: Attention mask for the chunk.
            - `offset_mapping`: Character start and end offsets for tokens in
              the original text.
    """
    enc = tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        return_offsets_mapping=True,
        return_token_type_ids=False,
    )

    chunks = []
    for i in range(len(enc["input_ids"])):
        chunks.append(
            {
                "input_ids": enc["input_ids"][i],
                "attention_mask": enc["attention_mask"][i],
                "offset_mapping": enc["offset_mapping"][i],
            }
        )
    return chunks


@torch.no_grad()
def predict_chunks(chunks: list[dict[str, Any]]) -> tuple[np.ndarray, np.ndarray]:
    """Predict class probabilities and labels for tokenized chunks.

    Runs each tokenized chunk through the global `model` using the global
    `DEVICE`, then computes softmax probabilities from the model logits. The
    predicted label for each chunk is the index of the largest probability.

    Args:
        chunks: Tokenized chunks. Each chunk must contain `input_ids` and
            `attention_mask` entries that can be converted to PyTorch tensors.

    Returns:
        A tuple containing:
            - A NumPy array of class probabilities with shape
              `(num_chunks, num_classes)`.
            - A NumPy array of predicted class indices with shape
              `(num_chunks,)`.
    """
    all_probs = []
    all_preds = []
    for chunk in chunks:
        inputs = {
            "input_ids": torch.tensor([chunk["input_ids"]], device=DEVICE),
            "attention_mask": torch.tensor([chunk["attention_mask"]], device=DEVICE),
        }
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).detach().cpu().numpy()[0]
        pred = int(np.argmax(probs))
        all_probs.append(probs)
        all_preds.append(pred)
    return np.array(all_probs), np.array(all_preds)


def predict_document(
    text: str,
    max_length: int = 256,
    stride: int = 64,
) -> dict[str, Any]:
    """Predict a document label by averaging predictions across text chunks.

    Splits `text` into tokenized chunks, predicts class probabilities for each
    chunk, and averages the chunk-level probabilities to produce a document-level
    prediction. The function also builds a DataFrame containing per-chunk
    predictions, confidence scores, and class probabilities.

    Args:
        text: Document text to classify.
        max_length: Maximum number of tokens per chunk, including special
            tokens.
        stride: Number of overlapping tokens to keep between consecutive chunks.

    Returns:
        A dictionary containing:
            - `chunks`: Tokenized chunks produced from the input text.
            - `chunk_df`: DataFrame with one row per chunk, including predicted
              labels, confidence scores, and per-label probabilities.
            - `doc_probs`: Averaged document-level class probabilities.
            - `doc_pred_id`: Predicted document-level class index.
            - `doc_pred_label`: Predicted document-level class label.
    """
    chunks = chunk_document(text, tokenizer, max_length=max_length, stride=stride)
    probs, preds = predict_chunks(chunks)

    avg_probs = probs.mean(axis=0)
    pred = int(np.argmax(avg_probs))

    rows = []
    for i, p in enumerate(probs):
        row = {
            "chunk_id": i,
            "pred_label": id2label[int(preds[i])],
            "pred_confidence": float(np.max(p)),
        }
        for j in range(len(p)):
            row[f"prob_{id2label[j]}"] = float(p[j])
        rows.append(row)

    return {
        "chunks": chunks,
        "chunk_df": pd.DataFrame(rows),
        "doc_probs": avg_probs,
        "doc_pred_id": pred,
        "doc_pred_label": id2label[pred],
    }


def pretty_probabilities(prob_vector: Sequence[float]) -> pd.DataFrame:
    """Return class probabilities as a sorted DataFrame.

    Converts a probability vector into a DataFrame with one row per class label.
    Labels are resolved from the global `id2label` mapping using each
    probability's index. Rows are sorted by probability in descending order.

    Args:
        prob_vector: Sequence of class probabilities, where each position
            corresponds to a class index in `id2label`.

    Returns:
        A DataFrame with `label` and `probability` columns, sorted from highest
        to lowest probability.
    """
    rows = []
    for i, p in enumerate(prob_vector):
        rows.append({"label": id2label[i], "probability": float(p)})
    return pd.DataFrame(rows).sort_values("probability", ascending=False).reset_index(drop=True)


def forward_func(input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """Run the model forward pass and return logits.

    Calls the global `model` with token IDs and an attention mask, then returns
    the logits from the model output.

    Args:
        input_ids: Tensor containing token IDs to pass to the model.
        attention_mask: Tensor indicating which tokens should be attended to.

    Returns:
        The logits tensor produced by the model.
    """
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    return outputs.logits


def get_target_id(target_label: Optional[int | str], doc_pred_id: int) -> int:
    """Resolve a target class ID from a label, class ID, or document prediction.

    If `target_label` is `None`, the document-level prediction ID is returned.
    If `target_label` is already an integer, it is returned as an integer.
    Otherwise, `target_label` is treated as a label name and resolved through
    the global `label2id` mapping.

    Args:
        target_label: Target class label name, target class ID, or `None` to use
            the document-level prediction ID.
        doc_pred_id: Document-level predicted class ID used when `target_label`
            is `None`.

    Returns:
        The resolved target class ID.
    """
    if target_label is None:
        return int(doc_pred_id)
    if isinstance(target_label, int):
        return int(target_label)
    return int(label2id[target_label])


def summarize_attributions(attributions: torch.Tensor) -> np.ndarray:
    """Summarize token attributions across the embedding dimension.

    Sums attribution scores over the final tensor dimension, removes a leading
    batch dimension of size one when present, and normalizes the resulting
    attribution vector by its L2 norm when the norm is nonzero.

    Args:
        attributions: Attribution tensor whose final dimension represents
            embedding-level attribution values.

    Returns:
        A NumPy array containing normalized attribution scores. If the L2 norm is
        zero, the unnormalized summed attributions are returned as a NumPy array.
    """
    # Sum over the embedding dimension.
    attributions = attributions.sum(dim=-1).squeeze(0)
    norm = torch.norm(attributions)
    if norm.item() != 0:
        attributions = attributions / norm
    return attributions.detach().cpu().numpy()


def explain_chunk(
    chunk: dict[str, Any],
    lig: LayerIntegratedGradients,
    target_id: int | None = None,
    n_steps: int = 50,
) -> dict[str, Any]:
    """Explain token attributions for a tokenized chunk.

    Computes layer integrated gradients for a single tokenized chunk using the
    global `lig`, `tokenizer`, and `DEVICE` dependencies. The baseline input is
    filled with the tokenizer PAD token ID. If the tokenizer has no PAD token
    ID, `1` is used as a fallback. The resulting attributions are summarized per
    token and returned with a sorted attribution table.

    Args:
        chunk: Tokenized chunk containing an `input_ids` entry and an
            `attention_mask` entry.
        lig: LayerIntegratedGradients instance to use for attribution.
        target_id: Optional target class ID for attribution. When `None`, the
            attribution method uses its default target behavior.
        n_steps: Number of integration steps used by layer integrated gradients.

    Returns:
        A dictionary containing:
            - `tokens`: Token strings corresponding to the input IDs.
            - `scores`: NumPy array of summarized attribution scores.
            - `delta`: NumPy array containing the convergence delta returned by
              the attribution method.
            - `table`: DataFrame sorted by absolute attribution magnitude, with
              token, attribution, and absolute attribution columns.
    """
    input_ids = torch.tensor([chunk["input_ids"]], device=DEVICE)
    attention_mask = torch.tensor([chunk["attention_mask"]], device=DEVICE)

    # Use PAD token IDs as the attribution baseline.
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = 1

    baseline_ids = torch.full_like(input_ids, fill_value=pad_id)

    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        additional_forward_args=(attention_mask,),
        target=target_id,
        return_convergence_delta=True,
        n_steps=n_steps,
    )

    token_list = tokenizer.convert_ids_to_tokens(input_ids[0].detach().cpu().tolist())
    scores = summarize_attributions(attributions)

    df = (
        pd.DataFrame(
            {
                "token": token_list,
                "attribution": scores,
                "abs_attribution": np.abs(scores),
            }
        )
        .sort_values("abs_attribution", ascending=False)
        .reset_index(drop=True)
    )

    return {
        "tokens": token_list,
        "scores": scores,
        "delta": delta.detach().cpu().numpy(),
        "table": df,
    }


def merge_roberta_tokens(tokens: Sequence[str]) -> tuple[list[str], list[list[int]]]:
    """Merge RoBERTa-style tokens into word-like strings.

    Groups tokens that belong to the same word based on RoBERTa's `Ġ` prefix,
    which marks tokens that begin after a space. Special tokens from the global
    `tokenizer` are kept as standalone tokens.

    Args:
        tokens: Sequence of token strings to merge.

    Returns:
        A tuple containing:
            - A list of merged word-like strings and standalone special tokens.
            - A list of token index groups. Each group contains the original
              token indices that contributed to the corresponding merged string.
    """
    words = []
    idx_groups = []
    current = ""
    current_group = []

    for i, tok in enumerate(tokens):
        if tok in tokenizer.all_special_tokens:
            if current:
                words.append(current)
                idx_groups.append(current_group)
                current = ""
                current_group = []
            words.append(tok)
            idx_groups.append([i])
            continue

        clean_tok = tok.replace("Ġ", " ")
        if tok.startswith("Ġ"):
            if current:
                words.append(current)
                idx_groups.append(current_group)
            current = clean_tok
            current_group = [i]
        else:
            current += clean_tok
            current_group.append(i)

    if current:
        words.append(current)
        idx_groups.append(current_group)

    return words, idx_groups


def aggregate_word_scores(
    tokens: Sequence[str],
    scores: Sequence[float],
) -> tuple[list[str], list[float]]:
    """Aggregate token attribution scores into word-level scores.

    Merges RoBERTa-style tokens into word-like strings using
    `merge_roberta_tokens`, then sums the attribution scores for the token
    indices that contributed to each merged word.

    Args:
        tokens: Sequence of token strings to merge and score.
        scores: Sequence of token-level scores. Each score is expected to align
            by index with `tokens`.

    Returns:
        A tuple containing:
            - Merged word-like strings.
            - Word-level scores computed as the sum of token-level scores for
              each merged word.
    """
    words, idx_groups = merge_roberta_tokens(tokens)
    word_scores = []
    for group in idx_groups:
        word_scores.append(float(np.sum([scores[i] for i in group])))
    return words, word_scores


def colorize_text(
    words: Sequence[str],
    scores: Sequence[float],
    max_words: int = 300,
) -> HTML:
    """Return HTML markup that colorizes words by attribution score.

    Clips the input to at most `max_words` items, then renders each word as an
    HTML span. Positive scores are shown with a red background, negative scores
    with a blue background, and color intensity is scaled by the largest absolute
    score in the clipped scores.

    Args:
        words: Sequence of words or tokens to display.
        scores: Sequence of attribution scores aligned by index with `words`.
        max_words: Maximum number of words and scores to render.

    Returns:
        An IPython HTML object containing colorized word spans.
    """
    clipped_words = words[:max_words]
    clipped_scores = scores[:max_words]
    max_abs = max(max(abs(s) for s in clipped_scores), 1e-8)

    spans = []
    for word, score in zip(clipped_words, clipped_scores):
        intensity = min(abs(score) / max_abs, 1.0)
        if score >= 0:
            bg = f"rgba(255, 80, 80, {0.15 + 0.55 * intensity})"
        else:
            bg = f"rgba(80, 120, 255, {0.15 + 0.55 * intensity})"

        safe_word = word.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        spans.append(
            f'<span title="{score:.4f}" style="background:{bg}; padding:2px 3px; margin:1px; border-radius:4px; display:inline-block;">{safe_word}</span>'
        )

    return HTML("<div style='line-height:2.0'>" + " ".join(spans) + "</div>")

## Obtener predicciones

In [7]:
text = get_text(TEXT_PATH, PASTED_TEXT)
result = predict_document(text, max_length=MAX_LENGTH, stride=STRIDE)

print("Autor predicho:", result["doc_pred_label"])
print("Probabilidades:")
pretty_probabilities(result["doc_probs"])

Autor predicho: arthur_conan_doyle
Probabilidades:


,label,probability
0,arthur_conan_doyle,0.989947
1,wilkie_collins,0.005010
2,richard_austin_freeman,0.002510
3,anna_katharine_green,0.002507
4,arthur_morrison,0.000015
5,gilbert_keith_chesterton,0.000012


In [8]:
print("Fragmentos:")
result["chunk_df"].head(10)

Fragmentos:


,chunk_id,pred_label,pred_confidence,prob_anna_katharine_green,prob_arthur_conan_doyle,prob_arthur_morrison,prob_gilbert_keith_chesterton,prob_richard_austin_freeman,prob_wilkie_collins
0,0,arthur_conan_doyle,0.999958,0.000008,0.999958,0.000009,0.000011,0.000007,0.000008
1,1,arthur_conan_doyle,0.999958,0.000008,0.999958,0.000009,0.000010,0.000007,0.000007
2,2,arthur_conan_doyle,0.999958,0.000008,0.999958,0.000009,0.000010,0.000007,0.000007
3,3,arthur_conan_doyle,0.999956,0.000008,0.999956,0.000009,0.000011,0.000008,0.000008
4,4,arthur_conan_doyle,0.999958,0.000008,0.999958,0.000009,0.000010,0.000007,0.000008
5,5,arthur_conan_doyle,0.999958,0.000007,0.999958,0.000009,0.000010,0.000007,0.000008
6,6,arthur_conan_doyle,0.999952,0.000006,0.999952,0.000013,0.000013,0.000009,0.000008
7,7,arthur_conan_doyle,0.999957,0.000007,0.999957,0.000009,0.000011,0.000008,0.000008
8,8,arthur_conan_doyle,0.999958,0.000007,0.999958,0.000009,0.000011,0.000007,0.000008
9,9,arthur_conan_doyle,0.999275,0.000006,0.999275,0.000077,0.000034,0.000577,0.000032


## *Integrated Gradients* sobre *embeddings*

A continuación, vamoa a preparar la explicación de la predicción del modelo mediante `LayerIntegratedGradients` y seleccionando qué fragmentos del documento serán explicados.

Para ello, primero se obtiene la capa de *embeddings* del modelo. En un modelo de texto, esta capa es la encargada de transformar los identificadores de los tokens en vectores numéricos que el modelo puede procesar.

Después se crea el objeto `LayerIntegratedGradients`, indicando dos elementos: la función `forward_func`, que define cómo se obtiene la salida del modelo que se quiere explicar, y `embedding_layer`, que es la capa sobre la que se calcularán las atribuciones. En este caso, las explicaciones se calcularán a nivel de la representación interna de los tokens, no directamente sobre los identificadores discretos de entrada.

In [9]:
embedding_layer = model.get_input_embeddings()
lig = LayerIntegratedGradients(forward_func, embedding_layer)

A continuación, el código determina qué clase se va a explicar. Para ello recupera de `result` la clase predicha para el documento completo, almacenada en `doc_pred_id`.

Luego, mediante `get_target_id`, se decide cuál será la clase objetivo de la explicación. Esta función permite elegir entre explicar la clase predicha por el modelo o una clase concreta definida en `TARGET_LABEL`. El identificador numérico de esa clase se convierte después en una etiqueta legible usando `id2label`.

In [10]:
doc_pred_id = result["doc_pred_id"]
target_id = get_target_id(TARGET_LABEL, doc_pred_id)
target_label = id2label[target_id]

print("Clase explicada:", target_label)

chunk_df = result["chunk_df"].copy()
target_prob_col = f"prob_{target_label}"

top_chunk_ids = chunk_df.sort_values(target_prob_col, ascending=False).head(TOP_CHUNKS_TO_EXPLAIN)["chunk_id"].tolist()

print("Chunks seleccionados:", top_chunk_ids)

chunk_df.sample(10)

Clase explicada: arthur_conan_doyle
Chunks seleccionados: [54, 309, 351]


,chunk_id,pred_label,pred_confidence,prob_anna_katharine_green,prob_arthur_conan_doyle,prob_arthur_morrison,prob_gilbert_keith_chesterton,prob_richard_austin_freeman,prob_wilkie_collins
334,334,arthur_conan_doyle,0.999958,0.000008,0.999958,0.000009,0.000010,0.000007,0.000008
172,172,arthur_conan_doyle,0.999958,0.000008,0.999958,0.000009,0.000010,0.000007,0.000008
202,202,arthur_conan_doyle,0.999958,0.000009,0.999958,0.000009,0.000010,0.000006,0.000008
272,272,arthur_conan_doyle,0.999958,0.000008,0.999958,0.000009,0.000010,0.000007,0.000008
162,162,arthur_conan_doyle,0.999948,0.000008,0.999948,0.000012,0.000014,0.000010,0.000009
76,76,arthur_conan_doyle,0.999956,0.000007,0.999956,0.000010,0.000012,0.000007,0.000008
1,1,arthur_conan_doyle,0.999958,0.000008,0.999958,0.000009,0.000010,0.000007,0.000007
223,223,arthur_conan_doyle,0.999958,0.000008,0.999958,0.000009,0.000010,0.000007,0.000008
356,356,arthur_conan_doyle,0.999959,0.000008,0.999959,0.000009,0.000010,0.000007,0.000008
290,290,arthur_conan_doyle,0.999958,0.000008,0.999958,0.000009,0.000010,0.000007,0.000008


Después se copia el DataFrame que contiene la información asociada a los distintos *chunks* o fragmentos del documento. Cada *chunk* representa una parte del texto original que ha sido evaluada por el modelo.

En resumen, el código anterior hace tres cosas principales: inicializa el método de explicabilidad LayerIntegratedGradients, determina qué autor o clase se va a explicar, y selecciona los fragmentos del texto que más apoyan esa predicción para analizarlos posteriormente con Captum.

A continuación, recorremos los *chunks* seleccionados previamente y, para cada uno de ellos, calcularemos una explicación con `LayerIntegratedGradients` mediante la función `explain_chunk`. La explicación se genera para la clase objetivo definida por `target_id`, es decir, el autor o etiqueta cuya predicción se quiere interpretar.

Después, las atribuciones obtenidas a nivel de token se agregan a nivel de palabra mediante `aggregate_word_scores`, lo que facilita una interpretación más legible del texto. Los resultados de cada fragmento se guardan en la lista `explanations`, junto con el identificador del *chunk*, la explicación completa, las palabras reconstruidas y sus puntuaciones.

Finalmente, el código muestra una tabla con las principales atribuciones y una visualización coloreada del texto, permitiendo identificar qué palabras o fragmentos han contribuido más positiva o negativamente a que el modelo asigne el texto al autor explicado.

In [11]:
explanations = []

for chunk_id in top_chunk_ids:
    chunk = result["chunks"][chunk_id]
    exp = explain_chunk(chunk, lig, target_id=target_id, n_steps=50)
    words, word_scores = aggregate_word_scores(exp["tokens"], exp["scores"])

    explanations.append(
        {
            "chunk_id": chunk_id,
            "exp": exp,
            "words": words,
            "word_scores": word_scores,
        }
    )

    print("\n" + "=" * 80)
    print(f"CHUNK {chunk_id} | Clase explicada: {target_label}")
    print("=" * 80)
    display(exp["table"].head(20))
    display(colorize_text(words, word_scores, max_words=250))


CHUNK 54 | Clase explicada: arthur_conan_doyle


,token,attribution,abs_attribution
0,ĠWatson,0.402775,0.402775
1,ĠHolmes,0.264122,0.264122
2,Ġupon,0.231270,0.231270
3,.,0.179261,0.179261
4,.,0.178992,0.178992
5,"?""",0.147236,0.147236
6,Ġshould,-0.145753,0.145753
7,Ġthat,0.137542,0.137542
8,.,0.135824,0.135824
9,<s>,0.132158,0.132158



CHUNK 309 | Clase explicada: arthur_conan_doyle


,token,attribution,abs_attribution
0,ĠHolmes,0.611429,0.611429
1,.,0.264279,0.264279
2,ĠIt,0.219038,0.219038
3,Ġupon,0.204471,0.204471
4,Ġmoon,0.185148,0.185148
5,Ġwhich,0.176806,0.176806
6,<s>,0.172310,0.172310
7,Ġwas,0.155839,0.155839
8,.,0.144537,0.144537
9,Ġrock,0.142566,0.142566



CHUNK 351 | Clase explicada: arthur_conan_doyle


,token,attribution,abs_attribution
0,Ġupon,0.392185,0.392185
1,ĠWatson,0.390921,0.390921
2,ĠHolmes,0.342945,0.342945
3,<s>,0.203755,0.203755
4,Ġgreat,0.185500,0.185500
5,Ġits,0.185229,0.185229
6,Ġmoon,0.177786,0.177786
7,.,0.164170,0.164170
8,It,0.150058,0.150058
9,.,0.142155,0.142155


## Explicar un *chunk* concreto manualmente

Aquí hacemos el mismo proceso pero para un *chunk* concreto, sin necesidad de seleccionar previamente los más relevantes. Esto permite analizar cualquier fragmento del documento, aunque no sea el que más apoye la predicción del modelo.

In [12]:
MANUAL_CHUNK_ID = 0

manual_chunk = result["chunks"][MANUAL_CHUNK_ID]
manual_exp = explain_chunk(manual_chunk, lig, target_id=target_id, n_steps=50)
manual_words, manual_word_scores = aggregate_word_scores(manual_exp["tokens"], manual_exp["scores"])

display(manual_exp["table"].head(25))
display(colorize_text(manual_words, manual_word_scores, max_words=250))

,token,attribution,abs_attribution
0,ĠWatson,0.419849,0.419849
1,Ġupon,0.372824,0.372824
2,.,0.242126,0.242126
3,.,0.197541,0.197541
4,Ġback,0.166494,0.166494
5,Ġupon,0.164244,0.164244
6,Ġfriends,0.150155,0.150155
7,Ġacross,0.145074,0.145074
8,Ġhim,0.139802,0.139802
9,ĠJames,0.132708,0.132708


## Comparar con otra clase

Usar una `ALTERNATIVE_LABEL` sirve para comparar la explicación de la clase predicha con la explicación de otra clase candidata. En nuestro caso, si el modelo ha predicho que un fragmento pertenece a un autor determinado, la etiqueta alternativa permite analizar cómo cambiarían las atribuciones si en lugar de explicar ese autor se explicara otro. Es decir, no se pregunta solo “¿por qué el modelo ha elegido este autor?”, sino también “¿qué evidencias del texto apoyarían o no apoyarían a este otro autor?”.

Esto es útil para hacer un análisis comparativo entre autores. Por ejemplo, puede revelar que ciertas palabras, giros estilísticos o estructuras del fragmento contribuyen positivamente a la predicción del autor elegido, pero no a la de otro autor. También puede ayudar a entender casos ambiguos, errores de clasificación o autores con estilos parecidos. Si el modelo duda entre dos etiquetas, comparar la clase predicha con una `ALTERNATIVE_LABEL` permite ver qué rasgos del texto inclinan la decisión hacia una u otra clase.

In [13]:
if ALTERNATIVE_LABEL is not None:
    alt_id = get_target_id(ALTERNATIVE_LABEL, doc_pred_id)
    alt_exp = explain_chunk(result["chunks"][top_chunk_ids[0]], lig, target_id=alt_id, n_steps=50)
    alt_words, alt_word_scores = aggregate_word_scores(alt_exp["tokens"], alt_exp["scores"])

    print("Clase alternativa explicada:", id2label[alt_id])
    display(alt_exp["table"].head(25))
    display(colorize_text(alt_words, alt_word_scores, max_words=250))
else:
    print("Define ALTERNATIVE_LABEL si quieres comparar otra clase.")

Clase alternativa explicada: anna_katharine_green


,token,attribution,abs_attribution
0,ĠWatson,-0.293172,0.293172
1,.,-0.200896,0.200896
2,<s>,0.194846,0.194846
3,Ġimpression,0.185702,0.185702
4,Ġwhich,0.175513,0.175513
5,Ġthat,-0.172206,0.172206
6,Ġroom,0.162915,0.162915
7,Ġevidence,-0.162056,0.162056
8,Ġclub,0.151485,0.151485
9,ĠThrough,-0.151126,0.151126


## Guardar resultados

In [14]:
save_dir = Path(MODEL_DIR) / "explanations"
save_dir.mkdir(parents=True, exist_ok=True)

summary = {
    "model_dir": MODEL_DIR,
    "text_path": str(TEXT_PATH),
    "predicted_label": result["doc_pred_label"],
    "document_probabilities": {id2label[i]: float(result["doc_probs"][i]) for i in range(len(result["doc_probs"]))},
    "explained_label": target_label,
    "top_chunks_explained": top_chunk_ids,
}

with open(save_dir / "explanation_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

for item in explanations:
    out_df = pd.DataFrame({"word": item["words"], "score": item["word_scores"]})
    out_df.to_csv(save_dir / f"chunk_{item['chunk_id']}_word_attributions.csv", index=False)

print("Explicaciones guardadas en:", save_dir)

Explicaciones guardadas en: /content/drive/MyDrive/outputs/authorship_transformer/explanations
